# 02 — Preprocesamiento

Filtra el dataset a 3 clases, selecciona las features del sensor dorsal y guarda el CSV procesado.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from pathlib import Path

RAW_PATH = Path("../data/raw/posture_dataset.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

## 1. Carga

In [ ]:
df = pd.read_csv(RAW_PATH, index_col=0)
print(f"Dataset original: {df.shape}")

## 2. Filtrar a 3 clases y reasignar etiquetas

In [ ]:
# Mapeo: label original → label entrenamiento
# 2 (sitting)          → 0 (adequate)
# 4 (forward bending)  → 1 (forward_slouch)
# 5 (backward bending) → 2 (excessive_recline)
LABEL_MAP = {2.0: 0, 4.0: 1, 5.0: 2}
CLASS_NAMES = {0: "adequate", 1: "forward_slouch", 2: "excessive_recline"}

df_f = df[df["label"].isin(LABEL_MAP.keys())].copy()
df_f["label"] = df_f["label"].map(LABEL_MAP)

print(f"Shape después de filtrar: {df_f.shape}")
print("Distribución de clases:")
print(df_f["label"].map(CLASS_NAMES).value_counts())

## 3. Selección de features (modelo v1 — solo sensor dorsal)

In [ ]:
# El dataset tiene sensor chest (Ax1,Ay1,Az1) y thigh (Ax2,Ay2,Az2).
# El sensor chest corresponde al sensor dorsal del chaleco SitRight.
# El modelo v1 usa solo las 3 features del sensor dorsal (ADR-004).
FEATURES = ["Ax1", "Ay1", "Az1"]

X = df_f[FEATURES].copy()
y = df_f["label"].copy()

print(f"X shape: {X.shape}")
print(f"NaN en X: {X.isnull().sum().sum()}")
print(f"Inf en X: {np.isinf(X.values).sum()}")

## 4. Shuffle y guardado

In [ ]:
X, y = shuffle(X, y, random_state=42)
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)

processed = X.copy()
processed["label"] = y

out_path = PROCESSED_DIR / "train_data.csv"
processed.to_csv(out_path, index=False)

print(f"Guardado en: {out_path}")
print(f"Shape final: {processed.shape}")
print(f"Clases: {CLASS_NAMES}")
processed.head()

## Resumen

- **Features:** `Ax1, Ay1, Az1` (sensor dorsal)
- **Labels:** 0=adequate, 1=forward_slouch, 2=excessive_recline
- **Archivo generado:** `data/processed/train_data.csv`
- **Siguiente paso:** `03-training.ipynb`